# Exploring an EMRI injection

Load a config, build the injection and likelihood a PE run would use, and
interrogate them: SNR, log-likelihood, overlap, and diagnostic plots. Nothing
here writes to the run directory or starts a sampler.

Needs the lisatools and viz extras: `pip install -e .[lisatools,viz]`

In [ ]:
from emridispatch.cli import set_env_guards

set_env_guards()

%matplotlib inline

from emridispatch.workbench import (
    injection_template, load, lnlike, measure, noise, offset,
    prior_from_config, signal, snr, truth)
from emridispatch.workbench_plots import (
    plot_char_strain, plot_snr_accumulation, plot_time_domain,
    plot_time_frequency)

In [ ]:
# load an injection from a config file
cfg, model = load("plot_config.yaml")
model.optimal_snr, model.add_noise, model.noise_seed

## Measures at the injected truth

`measure` generates the waveform once and returns the following information:
- optimal and matched-filter SNRs
- truncated log-likelihood used in sampling
- full log-likelihood with data and noise terms
- overlap

Pass `per_channel=True` for the per-TDI-channel breakdown.

`snr.optimal` is noise-free by construction; `snr.detected`, `lnlike` and
`overlap` all involve the data and so include the noise realization when
`data.add_noise` is on.

In [ ]:
at_truth = measure(model, truth(model), per_channel=True)
print(f"optimal SNR:\t\t {at_truth.snr.optimal}")
print(f"matched-filter SNR:\t {at_truth.snr.detected}")
print(f"log-likelihood (sample): {at_truth.lnlike}")
print(f"log-likelihood:\t\t {at_truth.lnlike_full}")
print(f"overlap:\t\t {at_truth.overlap}")

## Perturbing one parameter

`offset` adds deltas in sampling coordinates, so `ln_m1` is a log offset while
`p` is linear. Reuse a template across measures by building it once with
`signal`.

In [ ]:
h = signal(model, offset(model, p=+0.01))
snr(model, h), lnlike(model, h)

In [ ]:
import numpy as np

for delta in [-3e-6, -2e-6, -1e-6, 0.0, 1e-6, 2e-6, 3e-6]:
    m = measure(model, offset(model, p=delta))
    print(f"dp = {delta:+.0e}  lnL = {m.lnlike:12.4f}  "
          f"overlap = {m.overlap:.4f}")

## The noise realization

`injection_template(model)` regenerates the noiseless injected signal, which gets cached
after the first call. `noise(model)` is `d - h_injection` or zeros when `data.add_noise` is off.

In [ ]:
n = noise(model)
np.abs(n).max(), np.abs(injection_template(model).data_res_arr.arr).max()

## Prior

Rebuilds the prior from the config file or cached prior files that have been 
saved during a sampling run. The `sef` Fisher path is gated behind
`allow_fisher=True` because it can take a while to run.

In [ ]:
prior = prior_from_config(cfg, model=model)
rng = np.random.default_rng(0)
draws = prior.sample(rng, size=5)
print([measure(model, d).snr.optimal for d in draws])

## Plots

Every plot takes an optional parameter set (`None` means the injection), a
`show` tuple naming which traces to overlay, and returns `(fig, axes)` without
writing files. Traces are `"template"`, `"injection"`, `"data"` (noisy when
`add_noise` is on) and `"noise"`. `stft_kwargs` goes to `scipy.signal.stft`;
`**kwargs` goes to the matplotlib artist.


In [ ]:
fig, axes = plot_char_strain(model, show=("noise", "data", "template"))

In [ ]:
fig, axes = plot_time_frequency(model, stft_kwargs={"nperseg": 512})

In [ ]:
fig, axes = plot_snr_accumulation(model, show=("template", "data"))

In [ ]:
fig, axes = plot_time_domain(model, offset(model, p=+0.01),
                             show=("noise", "template"))